In [1]:
import matplotlib.pyplot as plt
import numpy as np

In [2]:
from fetch_stac import fetch_s2
from utils import softclip

In [3]:
STAC_URL = "https://earth-search.aws.element84.com/v1"

In [4]:
BANDS = [
    "coastal",    # B1: Coastal Aerosol, 60 m/px, 443 nm, Atmospheric correction
    "blue",       # B2: Blue, 10 m/px, 490 nm, Forest type mapping
    "green",      # B3: Green, 10 m/px, 560 nm, Water body analysis
    "red",        # B4: Red, 10 m/px, 665 nm, Vegetation discrimination
    "rededge1",   # B5: Red Edge 1, 20 m/px, 705 nm, Vegetation stress detection
    "rededge2",   # B6: Red Edge 2, 20 m/px, 740 nm, Leaf area index monitoring
    "rededge3",   # B7: Red Edge 3, 20 m/px, 783 nm, Biomass estimation
    "nir",        # B8: NIR, 10 m/px, 842 nm, Shorelines and biomass
    "nir08",      # B8A: Narrow NIR, 20 m/px, 865 nm, Vegetation classification
    "nir09",      # B9: Water Vapour, 60 m/px, 945 nm, Atmospheric water vapour
    "swir16",     # B11: SWIR 1, 20 m/px, 1610 nm, Soil and plant moisture
    "swir22",     # B12: SWIR 2, 20 m/px, 2190 nm, Geology and burn scars
]

GROUPS = {
    "Visible 1":  ["true_colour", "coastal"],
    "Visible 2":  ["blue", "green", "red"],
    "Red edge": ["rededge1", "rededge2", "rededge3"],
    "NIR":      ["nir", "nir08", "nir09"],
    "SWIR":     ["swir16", "swir22"],
}

LABELS = {
    "true_colour": "True Colour: RGB Composite\n(10 m/px)\nNatural visualization",
    "coastal": "B1: Coastal Aerosol\n(60 m/px, 443 nm)\nAtmospheric correction",
    "blue": "B2: Blue\n(10 m/px, 490 nm)\nForest type mapping",
    "green": "B3: Green\n(10 m/px, 560 nm)\nWater body analysis",
    "red": "B4: Red\n(10 m/px, 665 nm)\nVegetation discrimination",
    "rededge1": "B5: Red Edge 1\n(20 m/px, 705 nm)\nVegetation stress detection",
    "rededge2": "B6: Red Edge 2\n(20 m/px, 740 nm)\nLeaf area index monitoring",
    "rededge3": "B7: Red Edge 3\n(20 m/px, 783 nm)\nBiomass estimation",
    "nir": "B8: NIR\n(10 m/px, 842 nm)\nShorelines and biomass",
    "nir08": "B8A: Narrow NIR\n(20 m/px, 865 nm)\nVegetation classification",
    "nir09": "B9: Water Vapour\n(60 m/px, 945 nm)\nAtmospheric water vapour",
    "swir16": "B11: SWIR 1\n(20 m/px, 1610 nm)\nSoil and plant moisture",
    "swir22": "B12: SWIR 2\n(20 m/px, 2190 nm)\nGeology and burn scars",
}

In [5]:
# Forêt de Haguenau
longitude = 7.8389
latitude = 48.8577
size_meters = 2000.0
max_cloud = 10
collection = "sentinel-2-l2a",
scale_factor = 10_000.0  # For S2
query = ["cloud_cover"]
date_start = "2025-09-01"
date_end = "2026-09-01"

In [6]:
img, date = fetch_s2(
    longitude,
    latitude,
    size_meters,
    date_start,
    date_end,    
    bands=BANDS,
    stac_url=STAC_URL,
    collection=collection,
    max_cloud=max_cloud,
    query=query,
    scale_factor=scale_factor,
)

ValueError: dictionary update sequence element #0 has length 11; 2 is required

In [ ]:
print(f"{date}: array {img.shape} (bands, H, W), reflectance {img.min():.3f} to {img.max():.3f}")

In [ ]:
print(f"{'band':10s} {'p2':>6s} {'p50':>6s} {'p90':>6s} {'p98':>6s} {'%zero':>6s}")
for name, b in zip(BANDS, img):
    p2, p50, p90, p98 = np.percentile(b, (2, 50, 90, 98))
    print(f"{name:10s} {p2:6.3f} {p50:6.3f} {p90:6.3f} {p98:6.3f} {100 * (b == 0).mean():6.1f}")

In [ ]:
idx = {name: i for i, name in enumerate(BANDS)}

nb_columns = max(len(names) for names in GROUPS.values())
fig, axes = plt.subplots(len(GROUPS), nb_columns, figsize=(3 * nb_columns, 3.2 * len(GROUPS)), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
 
for r, (group, names) in enumerate(GROUPS.items()):
    for c, name in enumerate(names):
        if name == "true_colour":
            rgb = img[[idx["red"], idx["green"], idx["blue"]]].transpose(1, 2, 0)
            hi = np.percentile(rgb, 99)
            axes[r, c].imshow(np.clip(rgb / hi, 0, 1) ** 0.7) 
        else:            
            band_data = img[idx[name]]
            axes[r, c].imshow(softclip(band_data), cmap="gray")
            
        axes[r, c].set_title(LABELS[name], fontsize=9)
        
    axes[r, 0].text(-0.06, 0.5, group, transform=axes[r, 0].transAxes, rotation=90, va="center", ha="right", fontsize=12, fontweight="bold")
 
fig.suptitle(f"Sentinel-2, {date}, lon {longitude} lat {latitude}, {size_meters:.0f} m")
plt.tight_layout()
plt.show()